# Fast & Slow Pointers (Floyd's Cycle Detection)

---

## What is the Fast & Slow Pointers Pattern?

Two pointers move through a sequence at **different speeds** — typically `slow` moves 1 step at a time and `fast` moves 2 steps. This creates a clever way to detect cycles, find midpoints, and solve many linked list problems in O(1) space.

---

## Core Uses

| Use Case | Mechanism |
|---|---|
| Detect cycle | If fast & slow ever meet → cycle exists |
| Find cycle start | After meeting, reset one pointer to head; advance both 1 step at a time |
| Find middle | When fast reaches end, slow is at middle |
| Find kth from end | Advance fast k steps, then move both together |

---

## ListNode Definition

```python
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next
```

---

## Core Templates

### Cycle Detection
```python
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast:
            return True
    return False
```

### Find Middle
```python
def find_middle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow  # slow is at middle
```

### Find Cycle Start (Floyd's)
```python
def cycle_start(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast:
            # Reset one pointer to head
            slow = head
            while slow != fast:
                slow = slow.next
                fast = fast.next
            return slow
    return None
```

**Why does cycle start work?** When fast and slow meet, the distance from the meeting point back to the cycle start equals the distance from the head to the cycle start. Proven by algebra on the cycle length.

---

## Complexity
| | Space |
|---|---|
| All fast/slow patterns | O(n) time, **O(1) space** |

In [ ]:
# Helper class used throughout this notebook
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def make_list(values):
    """Helper: create linked list from a list of values"""
    dummy = ListNode(0)
    curr = dummy
    for v in values:
        curr.next = ListNode(v)
        curr = curr.next
    return dummy.next

def to_list(head):
    """Helper: convert linked list to Python list"""
    result = []
    while head:
        result.append(head.val)
        head = head.next
    return result

print("Helper classes defined.")

---
# EASY Problems
---

## Easy 1 — Linked List Cycle (LeetCode 141)

> 🏢 **Asked by:** Amazon, Microsoft, Google, Meta
Detect if a linked list has a cycle.

### Approach
Move slow 1 step and fast 2 steps. If they ever meet, there's a cycle. If fast reaches None, no cycle.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def hasCycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast:
            return True
    return False

# No cycle
head = make_list([3, 2, 0, -4])
assert hasCycle(head) == False

# Create a cycle: tail -> node at index 1
head = make_list([3, 2, 0, -4])
nodes = []
curr = head
while curr:
    nodes.append(curr)
    curr = curr.next
nodes[-1].next = nodes[1]  # create cycle
assert hasCycle(head) == True
print("All tests passed!")

## Easy 2 — Middle of the Linked List (LeetCode 876)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Return the middle node. If two middles, return the second.

### Approach
When fast reaches the end (or None), slow is at the middle.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def middleNode(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

head = make_list([1, 2, 3, 4, 5])
assert middleNode(head).val == 3

head = make_list([1, 2, 3, 4, 5, 6])
assert middleNode(head).val == 4  # second middle
print("All tests passed!")

## Easy 3 — Palindrome Linked List (LeetCode 234)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Meta
Check if a linked list is a palindrome.

### Approach
1. Find middle using slow/fast.
2. Reverse the second half.
3. Compare first half and reversed second half.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def isPalindrome(head):
    # Step 1: find middle
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    # Step 2: reverse second half
    prev, curr = None, slow
    while curr:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
    # Step 3: compare
    left, right = head, prev
    while right:
        if left.val != right.val:
            return False
        left = left.next
        right = right.next
    return True

assert isPalindrome(make_list([1,2,2,1])) == True
assert isPalindrome(make_list([1,2])) == False
assert isPalindrome(make_list([1,2,3,2,1])) == True
print("All tests passed!")

## Easy 4 — Remove Nth Node From End of List (LeetCode 19)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Apple
Remove the nth node from the end in one pass.

### Approach
Advance `fast` n+1 steps ahead. Then move both until fast reaches None. `slow.next` is the node to remove.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def removeNthFromEnd(head, n):
    dummy = ListNode(0, head)
    slow = fast = dummy
    # advance fast n+1 steps
    for _ in range(n + 1):
        fast = fast.next
    # move both until fast is None
    while fast:
        slow = slow.next
        fast = fast.next
    slow.next = slow.next.next
    return dummy.next

assert to_list(removeNthFromEnd(make_list([1,2,3,4,5]), 2)) == [1,2,3,5]
assert to_list(removeNthFromEnd(make_list([1]), 1)) == []
assert to_list(removeNthFromEnd(make_list([1,2]), 1)) == [1]
print("All tests passed!")

## Easy 5 — Happy Number (LeetCode 202)

> 🏢 **Asked by:** Amazon, Google, Microsoft
A number is happy if repeated digit-squaring-sum eventually reaches 1. Detect cycles in the sequence.

### Approach
Treat the sequence of sums as a linked list. Use fast/slow to detect a cycle. If the cycle value is 1, it's happy.

**Time:** O(log n) | **Space:** O(1)

In [ ]:
def isHappy(n):
    def next_num(x):
        total = 0
        while x:
            x, digit = divmod(x, 10)
            total += digit ** 2
        return total

    slow, fast = n, next_num(n)
    while fast != 1 and slow != fast:
        slow = next_num(slow)
        fast = next_num(next_num(fast))
    return fast == 1

assert isHappy(19) == True
assert isHappy(2) == False
print("All tests passed!")

## Easy 6 — Merge Two Sorted Lists (LeetCode 21)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Apple, Bloomberg
Merge two sorted linked lists and return the merged list.

### Approach
Use a dummy head and a current pointer. At each step pick the smaller node from either list.

**Time:** O(n+m) | **Space:** O(1)

In [ ]:
def mergeTwoLists(l1, l2):
    dummy = ListNode(0)
    curr = dummy
    while l1 and l2:
        if l1.val <= l2.val:
            curr.next = l1
            l1 = l1.next
        else:
            curr.next = l2
            l2 = l2.next
        curr = curr.next
    curr.next = l1 or l2
    return dummy.next

assert to_list(mergeTwoLists(make_list([1,2,4]), make_list([1,3,4]))) == [1,1,2,3,4,4]
assert to_list(mergeTwoLists(make_list([]), make_list([]))) == []
print("All tests passed!")

## Easy 7 — Reverse Linked List (LeetCode 206)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Meta, Apple
Reverse a singly linked list.

### Approach
Iterative: use `prev`, `curr`, `next` pointers. At each step reverse the link and advance.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def reverseList(head):
    prev, curr = None, head
    while curr:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
    return prev

assert to_list(reverseList(make_list([1,2,3,4,5]))) == [5,4,3,2,1]
assert to_list(reverseList(make_list([1,2]))) == [2,1]
assert to_list(reverseList(make_list([]))) == []
print("All tests passed!")

## Easy 8 — Remove Duplicates from Sorted List (LeetCode 83)

> 🏢 **Asked by:** Amazon, Google
Delete all duplicate values, keeping only one occurrence.

### Approach
Scan with one pointer. When current and next have same value, skip next by redirecting the link.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def deleteDuplicates(head):
    curr = head
    while curr and curr.next:
        if curr.val == curr.next.val:
            curr.next = curr.next.next
        else:
            curr = curr.next
    return head

assert to_list(deleteDuplicates(make_list([1,1,2]))) == [1,2]
assert to_list(deleteDuplicates(make_list([1,1,2,3,3]))) == [1,2,3]
print("All tests passed!")

## Easy 9 — Convert Binary Number in Linked List to Integer (LeetCode 1290)

> 🏢 **Asked by:** Amazon, Google
The list represents a binary number (MSB first). Return its decimal value.

### Approach
Scan the list, building the number by left-shifting and OR-ing each bit.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def getDecimalValue(head):
    num = 0
    while head:
        num = (num << 1) | head.val
        head = head.next
    return num

assert getDecimalValue(make_list([1,0,1])) == 5
assert getDecimalValue(make_list([0])) == 0
assert getDecimalValue(make_list([1])) == 1
print("All tests passed!")

## Easy 10 — Delete Node in a Linked List (LeetCode 237)

> 🏢 **Asked by:** Amazon, Google
Delete a node given only access to that node (not the head).

### Approach
Copy the next node's value into the current node, then skip the next node.

**Time:** O(1) | **Space:** O(1)

In [ ]:
def deleteNode(node):
    node.val = node.next.val
    node.next = node.next.next

# Test: list [4,5,1,9], delete node with value 5
head = make_list([4,5,1,9])
node_to_delete = head.next  # node with value 5
deleteNode(node_to_delete)
assert to_list(head) == [4,1,9]
print("All tests passed!")

## Easy 11 — Intersection of Two Linked Lists (LeetCode 160)

> 🏢 **Asked by:** Amazon, Microsoft, Bloomberg
Find the node where two linked lists intersect.

### Approach
Use two pointers. When one reaches the end, redirect it to the head of the other list. After at most `len(A) + len(B)` steps, both pointers are at the same position.

**Time:** O(n+m) | **Space:** O(1)

In [ ]:
def getIntersectionNode(headA, headB):
    a, b = headA, headB
    while a != b:
        a = a.next if a else headB
        b = b.next if b else headA
    return a

# Build intersecting lists: [4,1] -> [8,4,5] and [5,6,1] -> [8,4,5]
common = make_list([8,4,5])
a = ListNode(4, ListNode(1, common))
b = ListNode(5, ListNode(6, ListNode(1, common)))
assert getIntersectionNode(a, b) == common

# No intersection
assert getIntersectionNode(make_list([2,6,4]), make_list([1,5])) is None
print("All tests passed!")

## Easy 12 — Swapping Nodes in a Linked List (LeetCode 1721)

> 🏢 **Asked by:** Amazon, Google
Swap values of the kth node from the start and kth node from the end.

### Approach
Find the kth node from front. Then use fast/slow: advance fast to end, slow starts at head — when fast reaches end, slow is at kth from end. Swap values.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def swapNodes(head, k):
    curr = head
    for _ in range(k - 1):
        curr = curr.next
    front = curr
    slow = head
    while curr.next:
        curr = curr.next
        slow = slow.next
    # slow is now kth from end
    front.val, slow.val = slow.val, front.val
    return head

assert to_list(swapNodes(make_list([1,2,3,4,5]), 2)) == [1,4,3,2,5]
assert to_list(swapNodes(make_list([7,9,6,6,7,8,3,0,9,5]), 5)) == [7,9,6,6,8,7,3,0,9,5]
print("All tests passed!")

## Easy 13 — Maximum Twin Sum of a Linked List (LeetCode 2130)

> 🏢 **Asked by:** Amazon, Google
For a linked list of even length, find the maximum sum of any pair (i, n-1-i).

### Approach
Find middle, reverse second half, walk both halves together taking the max sum of twins.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def pairSum(head):
    # find middle
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    # reverse second half
    prev, curr = None, slow
    while curr:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
    # compare twins
    max_sum = 0
    first, second = head, prev
    while second:
        max_sum = max(max_sum, first.val + second.val)
        first = first.next
        second = second.next
    return max_sum

assert pairSum(make_list([5,4,2,1])) == 6
assert pairSum(make_list([4,2,2,3])) == 7
assert pairSum(make_list([1,100000])) == 100001
print("All tests passed!")

## Easy 14 — Delete Middle Node of Linked List (LeetCode 2095)

> 🏢 **Asked by:** Amazon, Google
Delete the middle node of a linked list.

### Approach
Use slow/fast to find the node just before the middle. Then skip the middle node.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def deleteMiddle(head):
    if not head or not head.next:
        return None
    dummy = ListNode(0, head)
    slow, fast = dummy, head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    slow.next = slow.next.next
    return dummy.next

assert to_list(deleteMiddle(make_list([1,3,4,7,1,2,6]))) == [1,3,4,1,2,6]
assert to_list(deleteMiddle(make_list([1,2,3,4]))) == [1,2,4]
assert to_list(deleteMiddle(make_list([2,1]))) == [2]
print("All tests passed!")

## Easy 15 — Split Linked List in Parts (LeetCode 725)

> 🏢 **Asked by:** Amazon, Google
Split the linked list into k consecutive parts as evenly as possible.

### Approach
Count total length. Each part gets `length // k` nodes; the first `length % k` parts get one extra.

**Time:** O(n) | **Space:** O(k)

In [ ]:
def splitListToParts(head, k):
    length = 0
    curr = head
    while curr:
        length += 1
        curr = curr.next
    base, extra = divmod(length, k)
    result = []
    curr = head
    for i in range(k):
        part_head = curr
        part_len = base + (1 if i < extra else 0)
        for _ in range(part_len - 1):
            if curr:
                curr = curr.next
        if curr:
            nxt = curr.next
            curr.next = None
            curr = nxt
        result.append(part_head)
    return result

parts = splitListToParts(make_list([1,2,3,4,5,6,7,8,9,10]), 3)
assert [to_list(p) for p in parts] == [[1,2,3,4],[5,6,7],[8,9,10]]
print("All tests passed!")

## Easy 16 — Remove Linked List Elements (LeetCode 203)

> 🏢 **Asked by:** Amazon, Google
Remove all nodes with value equal to `val`.

### Approach
Use a dummy head to simplify edge cases. Scan and skip nodes with the target value.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def removeElements(head, val):
    dummy = ListNode(0, head)
    curr = dummy
    while curr.next:
        if curr.next.val == val:
            curr.next = curr.next.next
        else:
            curr = curr.next
    return dummy.next

assert to_list(removeElements(make_list([1,2,6,3,4,5,6]), 6)) == [1,2,3,4,5]
assert to_list(removeElements(make_list([7,7,7,7]), 7)) == []
print("All tests passed!")

## Easy 17 — Reverse Linked List (Recursive) (LeetCode 206)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Meta, Apple
Reverse using recursion.

### Approach
Recurse to the end. On the way back, reverse each link. The last node becomes the new head.

**Time:** O(n) | **Space:** O(n) call stack

In [ ]:
def reverseListRecursive(head):
    if not head or not head.next:
        return head
    new_head = reverseListRecursive(head.next)
    head.next.next = head  # reverse the link
    head.next = None
    return new_head

assert to_list(reverseListRecursive(make_list([1,2,3,4,5]))) == [5,4,3,2,1]
print("All tests passed!")

## Easy 18 — Count Nodes in Linked List

> 🏢 **Asked by:** Amazon, Google
Count the number of nodes in a linked list.

### Approach
Walk the list with a single pointer counting each node.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def countNodes(head):
    count = 0
    while head:
        count += 1
        head = head.next
    return count

assert countNodes(make_list([1,2,3,4,5])) == 5
assert countNodes(make_list([])) == 0
print("All tests passed!")

## Easy 19 — Linked List Cycle II — detect cycle start (LeetCode 142)

> 🏢 **Asked by:** Amazon, Microsoft, Google, Meta
Return the node where the cycle begins.

### Approach
After slow and fast meet inside the cycle, reset `slow` to head. Advance both 1 step at a time — they will meet at the cycle's start.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def detectCycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast:
            slow = head
            while slow != fast:
                slow = slow.next
                fast = fast.next
            return slow
    return None

# Build cycle: [3,2,0,-4] with tail connecting to node at index 1
head = make_list([3,2,0,-4])
nodes = []
curr = head
while curr:
    nodes.append(curr)
    curr = curr.next
nodes[-1].next = nodes[1]
assert detectCycle(head) == nodes[1]

assert detectCycle(make_list([1,2])) is None
print("All tests passed!")

## Easy 20 — Swap Nodes in Pairs (LeetCode 24)

> 🏢 **Asked by:** Amazon, Google
Swap every two adjacent nodes and return the head.

### Approach
Use a dummy node and process pairs iteratively: for each pair (a, b), set `prev.next = b`, `a.next = b.next`, `b.next = a`.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def swapPairs(head):
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next and prev.next.next:
        a = prev.next
        b = prev.next.next
        prev.next = b
        a.next = b.next
        b.next = a
        prev = a
    return dummy.next

assert to_list(swapPairs(make_list([1,2,3,4]))) == [2,1,4,3]
assert to_list(swapPairs(make_list([1,2,3]))) == [2,1,3]
print("All tests passed!")

---
# MEDIUM Problems
---

## Medium 1 — Reorder List (LeetCode 143)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Reorder L0 → Ln → L1 → Ln-1 → L2 → Ln-2 → … in-place.

### Approach
1. Find middle (fast/slow). 2. Reverse second half. 3. Merge the two halves alternately.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def reorderList(head):
    # Step 1: find middle
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    # Step 2: reverse second half
    prev, curr = None, slow.next
    slow.next = None  # cut the list
    while curr:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
    # Step 3: merge
    first, second = head, prev
    while second:
        tmp1, tmp2 = first.next, second.next
        first.next = second
        second.next = tmp1
        first = tmp1
        second = tmp2

head = make_list([1,2,3,4])
reorderList(head)
assert to_list(head) == [1,4,2,3]

head = make_list([1,2,3,4,5])
reorderList(head)
assert to_list(head) == [1,5,2,4,3]
print("All tests passed!")

## Medium 2 — Find the Duplicate Number (LeetCode 287)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Meta
Find the duplicate in an array of n+1 integers where each integer is in [1, n]. Use O(1) space.

### Approach
Treat the array as a linked list where `nums[i]` is the next node. Since there's a duplicate, there must be a cycle. Apply Floyd's cycle detection, then find the cycle start.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def findDuplicate(nums):
    slow = fast = nums[0]
    # Phase 1: detect cycle
    while True:
        slow = nums[slow]
        fast = nums[nums[fast]]
        if slow == fast:
            break
    # Phase 2: find cycle start
    slow = nums[0]
    while slow != fast:
        slow = nums[slow]
        fast = nums[fast]
    return slow

assert findDuplicate([1,3,4,2,2]) == 2
assert findDuplicate([3,1,3,4,2]) == 3
print("All tests passed!")

## Medium 3 — Rotate List (LeetCode 61)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Rotate a linked list to the right by k places.

### Approach
Find the length and make the list circular. Then find the new tail at position `n - k % n - 1` and break the cycle.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def rotateRight(head, k):
    if not head or not head.next or k == 0:
        return head
    # find length and tail
    length, tail = 1, head
    while tail.next:
        tail = tail.next
        length += 1
    k %= length
    if k == 0:
        return head
    tail.next = head  # make circular
    # new tail is at position length - k - 1 from head
    new_tail = head
    for _ in range(length - k - 1):
        new_tail = new_tail.next
    new_head = new_tail.next
    new_tail.next = None
    return new_head

assert to_list(rotateRight(make_list([1,2,3,4,5]), 2)) == [4,5,1,2,3]
assert to_list(rotateRight(make_list([0,1,2]), 4)) == [2,0,1]
print("All tests passed!")

## Medium 4 — Odd Even Linked List (LeetCode 328)

> 🏢 **Asked by:** Amazon, Google
Group all odd-indexed nodes together followed by even-indexed nodes.

### Approach
Use two pointers, one for odd indices and one for even. Build two separate chains, then link the end of the odd chain to the start of the even chain.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def oddEvenList(head):
    if not head:
        return head
    odd, even = head, head.next
    even_head = even
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = even_head
    return head

assert to_list(oddEvenList(make_list([1,2,3,4,5]))) == [1,3,5,2,4]
assert to_list(oddEvenList(make_list([2,1,3,5,6,4,7]))) == [2,3,6,7,1,5,4]
print("All tests passed!")

## Medium 5 — Add Two Numbers (LeetCode 2)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Bloomberg, Apple
Add two numbers represented as reversed linked lists.

### Approach
Simulate addition digit by digit with a carry. Use a dummy node to build the result list.

**Time:** O(max(n,m)) | **Space:** O(max(n,m))

In [ ]:
def addTwoNumbers(l1, l2):
    dummy = ListNode(0)
    curr = dummy
    carry = 0
    while l1 or l2 or carry:
        val = carry
        if l1:
            val += l1.val
            l1 = l1.next
        if l2:
            val += l2.val
            l2 = l2.next
        carry, val = divmod(val, 10)
        curr.next = ListNode(val)
        curr = curr.next
    return dummy.next

# 342 + 465 = 807  (stored reversed)
assert to_list(addTwoNumbers(make_list([2,4,3]), make_list([5,6,4]))) == [7,0,8]
assert to_list(addTwoNumbers(make_list([9,9,9,9,9,9,9]), make_list([9,9,9,9]))) == [8,9,9,9,0,0,0,1]
print("All tests passed!")

## Medium 6 — Partition List (LeetCode 86)

> 🏢 **Asked by:** Amazon, Google
Partition a linked list around value x: nodes less than x come before nodes ≥ x.

### Approach
Create two dummy heads: one for the 'less' partition and one for the 'greater or equal' partition. Fill both, then link them.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def partition(head, x):
    less_dummy = ListNode(0)
    great_dummy = ListNode(0)
    less, great = less_dummy, great_dummy
    curr = head
    while curr:
        if curr.val < x:
            less.next = curr
            less = less.next
        else:
            great.next = curr
            great = great.next
        curr = curr.next
    great.next = None
    less.next = great_dummy.next
    return less_dummy.next

assert to_list(partition(make_list([1,4,3,2,5,2]), 3)) == [1,2,2,4,3,5]
assert to_list(partition(make_list([2,1]), 2)) == [1,2]
print("All tests passed!")

## Medium 7 — Copy List with Random Pointer (LeetCode 138)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Bloomberg
Deep copy a linked list where each node has a `random` pointer.

### Approach
Use a hash map: old_node → new_node. First pass creates all new nodes. Second pass assigns `.next` and `.random` pointers.

**Time:** O(n) | **Space:** O(n)

In [ ]:
class NodeRandom:
    def __init__(self, x, next=None, random=None):
        self.val = x
        self.next = next
        self.random = random

def copyRandomList(head):
    if not head:
        return None
    mapping = {}
    curr = head
    while curr:
        mapping[curr] = NodeRandom(curr.val)
        curr = curr.next
    curr = head
    while curr:
        if curr.next:
            mapping[curr].next = mapping[curr.next]
        if curr.random:
            mapping[curr].random = mapping[curr.random]
        curr = curr.next
    return mapping[head]

# Build: [[7,null],[13,0],[11,4],[10,2],[1,0]]
nodes = [NodeRandom(v) for v in [7,13,11,10,1]]
for i in range(4): nodes[i].next = nodes[i+1]
nodes[1].random = nodes[0]
nodes[2].random = nodes[4]
nodes[3].random = nodes[2]
nodes[4].random = nodes[0]
copy = copyRandomList(nodes[0])
# Verify structure
vals = []
c = copy
while c:
    vals.append(c.val)
    c = c.next
assert vals == [7,13,11,10,1]
print("All tests passed!")

## Medium 8 — Sort List (LeetCode 148)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Meta
Sort a linked list in O(n log n) time using O(1) space.

### Approach
Bottom-up merge sort on linked list. Find middle (fast/slow), split, recursively sort, then merge. The fast/slow pattern is essential for finding the split point.

**Time:** O(n log n) | **Space:** O(log n) call stack

In [ ]:
def sortList(head):
    if not head or not head.next:
        return head
    # find middle and split
    slow, fast = head, head.next
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    mid = slow.next
    slow.next = None
    # sort both halves
    left = sortList(head)
    right = sortList(mid)
    # merge
    dummy = ListNode(0)
    curr = dummy
    while left and right:
        if left.val <= right.val:
            curr.next = left
            left = left.next
        else:
            curr.next = right
            right = right.next
        curr = curr.next
    curr.next = left or right
    return dummy.next

assert to_list(sortList(make_list([4,2,1,3]))) == [1,2,3,4]
assert to_list(sortList(make_list([-1,5,3,4,0]))) == [-1,0,3,4,5]
print("All tests passed!")

## Medium 9 — Remove Duplicates from Sorted List II (LeetCode 82)

> 🏢 **Asked by:** Amazon, Google
Delete all nodes with duplicate values, leaving only distinct numbers.

### Approach
Use a dummy head. When a run of duplicates is detected, skip the entire run.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def deleteDuplicatesII(head):
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        curr = prev.next
        # check if this is the start of a duplicate run
        if curr.next and curr.val == curr.next.val:
            # skip all nodes with this value
            val = curr.val
            while prev.next and prev.next.val == val:
                prev.next = prev.next.next
        else:
            prev = prev.next
    return dummy.next

assert to_list(deleteDuplicatesII(make_list([1,2,3,3,4,4,5]))) == [1,2,5]
assert to_list(deleteDuplicatesII(make_list([1,1,1,2,3]))) == [2,3]
print("All tests passed!")

## Medium 10 — Circular Array Loop (LeetCode 457)

> 🏢 **Asked by:** Amazon, Google
Determine if there is a cycle of length > 1 in a circular array where all steps in the cycle have the same direction.

### Approach
For each starting index, use fast/slow pointers. Validate that moves stay in the same direction and don't form a single-element cycle.

**Time:** O(n²) | **Space:** O(1)

In [ ]:
def circularArrayLoop(nums):
    n = len(nums)

    def next_idx(i):
        return (i + nums[i]) % n

    for start in range(n):
        slow, fast = start, start
        direction = nums[start] > 0
        while True:
            # validate slow step
            next_slow = next_idx(slow)
            if (nums[next_slow] > 0) != direction or next_slow == slow:
                break
            slow = next_slow
            # validate fast step (twice)
            next_fast = next_idx(fast)
            if (nums[next_fast] > 0) != direction or next_fast == fast:
                break
            next_fast2 = next_idx(next_fast)
            if (nums[next_fast2] > 0) != direction or next_fast2 == next_fast:
                break
            fast = next_fast2
            if slow == fast:
                return True
    return False

assert circularArrayLoop([2,-1,1,2,2]) == True
assert circularArrayLoop([-1,2]) == False
assert circularArrayLoop([-2,1,-1,-2,-2]) == False
print("All tests passed!")

## Medium 11 — Linked List Random Node (LeetCode 382)

> 🏢 **Asked by:** Amazon, Google
Return a random node's value from the linked list with equal probability.

### Approach
**Reservoir sampling**: iterate through the list. For the i-th node, replace the current result with probability 1/i. This ensures uniform probability without knowing the length.

**Time:** O(n) per getRandom | **Space:** O(1)

In [ ]:
import random

class LinkedListRandom:
    def __init__(self, head):
        self.head = head

    def getRandom(self):
        result = None
        curr = self.head
        i = 1
        while curr:
            if random.randint(1, i) == 1:
                result = curr.val
            curr = curr.next
            i += 1
        return result

ll = LinkedListRandom(make_list([1,2,3]))
results = {ll.getRandom() for _ in range(100)}
assert results == {1, 2, 3}  # all values should appear
print("All tests passed!")

## Medium 12 — Next Greater Node in Linked List (LeetCode 1019)

> 🏢 **Asked by:** Amazon, Google
For each node, find the value of the next greater node.

### Approach
Convert to array. Use a monotonic stack: push indices of unsatisfied elements. When a greater element is found, pop and fill in the answer.

**Time:** O(n) | **Space:** O(n)

In [ ]:
def nextLargerNodes(head):
    nums = []
    while head:
        nums.append(head.val)
        head = head.next
    result = [0] * len(nums)
    stack = []  # stores indices
    for i, val in enumerate(nums):
        while stack and nums[stack[-1]] < val:
            result[stack.pop()] = val
        stack.append(i)
    return result

assert nextLargerNodes(make_list([2,1,5])) == [5,5,0]
assert nextLargerNodes(make_list([2,7,4,3,5])) == [7,0,5,5,0]
print("All tests passed!")

## Medium 13 — Flatten a Multilevel Doubly Linked List (LeetCode 430)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Flatten a multilevel doubly linked list so all child lists are inlined.

### Approach
When we encounter a node with a child, insert the child list between the current node and its next. Use a stack or iterative tail-finding.

**Time:** O(n) | **Space:** O(1)

In [ ]:
class MLNode:
    def __init__(self, val, prev=None, next=None, child=None):
        self.val = val
        self.prev = prev
        self.next = next
        self.child = child

def flatten(head):
    curr = head
    while curr:
        if curr.child:
            child = curr.child
            nxt = curr.next
            # link curr -> child
            curr.next = child
            child.prev = curr
            curr.child = None
            # find tail of child list
            tail = child
            while tail.next:
                tail = tail.next
            # link tail -> nxt
            tail.next = nxt
            if nxt:
                nxt.prev = tail
        curr = curr.next
    return head

# Build: 1 <-> 2 <-> 3, with 2.child = 4 <-> 5
n1, n2, n3 = MLNode(1), MLNode(2), MLNode(3)
n4, n5 = MLNode(4), MLNode(5)
n1.next = n2; n2.prev = n1
n2.next = n3; n3.prev = n2
n4.next = n5; n5.prev = n4
n2.child = n4
result = flatten(n1)
vals = []
while result:
    vals.append(result.val)
    result = result.next
assert vals == [1, 2, 4, 5, 3]
print("All tests passed!")

## Medium 14 — LRU Cache (LeetCode 146)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg, Apple
Design an LRU cache with get and put operations in O(1) time.

### Approach
Combine a **doubly linked list** (for O(1) insertion/removal) with a **hash map** (for O(1) lookup). The list maintains order: most recently used at front, least recently used at back.

**Time:** O(1) for both operations | **Space:** O(capacity)

In [ ]:
class LRUCache:
    class Node:
        def __init__(self, key=0, val=0):
            self.key, self.val = key, val
            self.prev = self.next = None

    def __init__(self, capacity):
        self.cap = capacity
        self.map = {}
        self.head = self.Node()  # dummy head (most recent)
        self.tail = self.Node()  # dummy tail (least recent)
        self.head.next = self.tail
        self.tail.prev = self.head

    def _remove(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev

    def _insert_front(self, node):
        node.next = self.head.next
        node.prev = self.head
        self.head.next.prev = node
        self.head.next = node

    def get(self, key):
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node)
        self._insert_front(node)
        return node.val

    def put(self, key, value):
        if key in self.map:
            self._remove(self.map[key])
        node = self.Node(key, value)
        self._insert_front(node)
        self.map[key] = node
        if len(self.map) > self.cap:
            lru = self.tail.prev
            self._remove(lru)
            del self.map[lru.key]

cache = LRUCache(2)
cache.put(1, 1)
cache.put(2, 2)
assert cache.get(1) == 1
cache.put(3, 3)  # evicts key 2
assert cache.get(2) == -1
cache.put(4, 4)  # evicts key 1
assert cache.get(1) == -1
assert cache.get(3) == 3
assert cache.get(4) == 4
print("All tests passed!")

## Medium 15 — Remove Zero Sum Consecutive Nodes from Linked List (LeetCode 1171)

> 🏢 **Asked by:** Google, Amazon
Remove consecutive sequences of nodes that sum to zero.

### Approach
Use prefix sums. If two indices have the same prefix sum, the nodes between them sum to zero — skip them. Store prefix_sum → node in a dict; on second visit, update the pointer to skip the zero-sum run.

**Time:** O(n) | **Space:** O(n)

In [ ]:
def removeZeroSumSublists(head):
    dummy = ListNode(0, head)
    prefix_map = {0: dummy}
    prefix = 0
    curr = head
    while curr:
        prefix += curr.val
        prefix_map[prefix] = curr
        curr = curr.next
    # second pass: skip zero-sum runs
    prefix = 0
    curr = dummy
    while curr:
        prefix += curr.val
        curr.next = prefix_map[prefix].next
        curr = curr.next
    return dummy.next

assert to_list(removeZeroSumSublists(make_list([1,2,-3,3,1]))) == [3,1]
assert to_list(removeZeroSumSublists(make_list([1,2,3,-3,-2]))) == [1]
print("All tests passed!")

---
# HARD Problems
---

## Hard 1 — Reverse Nodes in k-Group (LeetCode 25)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft
Reverse every k nodes in the linked list. Leave remaining nodes as is.

### Approach
For each group of k nodes: check if k nodes exist, reverse them, link the reversed group back to the previous tail and forward to the next group.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def reverseKGroup(head, k):
    def reverse_segment(start, end):
        prev, curr = None, start
        while curr != end:
            nxt = curr.next
            curr.next = prev
            prev = curr
            curr = nxt
        return prev  # new head of reversed segment

    dummy = ListNode(0, head)
    group_prev = dummy
    while True:
        # check if k nodes remain
        kth = group_prev
        for _ in range(k):
            kth = kth.next
            if not kth:
                return dummy.next
        group_next = kth.next
        # reverse k nodes
        new_head = reverse_segment(group_prev.next, group_next)
        tail = group_prev.next  # original head is now tail
        group_prev.next = new_head
        tail.next = group_next
        group_prev = tail

assert to_list(reverseKGroup(make_list([1,2,3,4,5]), 2)) == [2,1,4,3,5]
assert to_list(reverseKGroup(make_list([1,2,3,4,5]), 3)) == [3,2,1,4,5]
print("All tests passed!")

## Hard 2 — Merge k Sorted Lists (LeetCode 23)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Merge k sorted linked lists into one sorted list.

### Approach
Use a **min-heap** of size k. Initialize with the head of each list. Pop the minimum, add it to the result, then push that node's next (if any).

**Time:** O(n log k) | **Space:** O(k)

In [ ]:
import heapq

def mergeKLists(lists):
    dummy = ListNode(0)
    curr = dummy
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
    while heap:
        val, i, node = heapq.heappop(heap)
        curr.next = node
        curr = curr.next
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next

lists = [make_list([1,4,5]), make_list([1,3,4]), make_list([2,6])]
assert to_list(mergeKLists(lists)) == [1,1,2,3,4,4,5,6]
print("All tests passed!")

## Hard 3 — Reverse Linked List II (LeetCode 92)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Meta, Apple
Reverse the linked list from position `left` to position `right`.

### Approach
Navigate to the node before `left`. Perform an in-place reversal of exactly `right - left` steps using the 'front insertion' technique.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def reverseBetween(head, left, right):
    dummy = ListNode(0, head)
    prev = dummy
    for _ in range(left - 1):
        prev = prev.next
    curr = prev.next
    for _ in range(right - left):
        nxt = curr.next
        curr.next = nxt.next
        nxt.next = prev.next
        prev.next = nxt
    return dummy.next

assert to_list(reverseBetween(make_list([1,2,3,4,5]), 2, 4)) == [1,4,3,2,5]
assert to_list(reverseBetween(make_list([5]), 1, 1)) == [5]
print("All tests passed!")

## Hard 4 — Linked List in Binary Tree (LeetCode 1367)

> 🏢 **Asked by:** Amazon, Google
Determine if a linked list starting from its head corresponds to a downward path in a binary tree.

### Approach
DFS on the tree. At each node, try to match the linked list from the beginning. A helper function checks if the list matches a downward path starting at a given tree node.

**Time:** O(n * m) | **Space:** O(n) call stack

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def isSubPath(head, root):
    def dfs_match(list_node, tree_node):
        if not list_node:
            return True  # matched the whole list
        if not tree_node:
            return False
        if list_node.val != tree_node.val:
            return False
        return dfs_match(list_node.next, tree_node.left) or \
               dfs_match(list_node.next, tree_node.right)

    if not root:
        return False
    # try starting a match at every tree node
    return dfs_match(head, root) or \
           isSubPath(head, root.left) or \
           isSubPath(head, root.right)

# Tree: [1,4,4,null,2,2,null,1,null,6,8]
# List: [4,2,8]
root = TreeNode(1)
root.left = TreeNode(4, TreeNode(2, TreeNode(1)), None)
root.right = TreeNode(4, TreeNode(2, TreeNode(6), TreeNode(8)), None)
head = make_list([4, 2, 8])
assert isSubPath(head, root) == True
print("All tests passed!")

## Hard 5 — Flatten Binary Tree to Linked List (LeetCode 114)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Bloomberg
Flatten a binary tree to a linked list in-place using preorder traversal.

### Approach
**Morris traversal style**: for each node that has a left child, find the rightmost node of the left subtree. Point it to the current node's right. Move the left subtree to the right and clear left.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def flattenTree(root):
    curr = root
    while curr:
        if curr.left:
            # find the rightmost of the left subtree
            rightmost = curr.left
            while rightmost.right:
                rightmost = rightmost.right
            # connect rightmost to current's right
            rightmost.right = curr.right
            curr.right = curr.left
            curr.left = None
        curr = curr.right

root = TreeNode(1)
root.left = TreeNode(2, TreeNode(3), TreeNode(4))
root.right = TreeNode(5, None, TreeNode(6))
flattenTree(root)
vals = []
curr = root
while curr:
    vals.append(curr.val)
    curr = curr.right
assert vals == [1,2,3,4,5,6]
print("All tests passed!")

## Hard 6 — Design a Skip List (simplified) (LeetCode 1206)

> 🏢 **Asked by:** Amazon, Google
Implement a skip list with add, erase, and search operations.

### Approach
A skip list is a probabilistic data structure where each node can appear in multiple levels of linked lists. Higher levels skip more nodes, allowing O(log n) search on average. Each node is promoted to the next level with probability 0.5.

**Time:** O(log n) average | **Space:** O(n log n) average

In [ ]:
import random

class Skiplist:
    MAX_LEVEL = 16

    class Node:
        def __init__(self, val, level):
            self.val = val
            self.next = [None] * level

    def __init__(self):
        self.head = self.Node(-1, self.MAX_LEVEL)
        self.level = 1

    def _random_level(self):
        lv = 1
        while random.random() < 0.5 and lv < self.MAX_LEVEL:
            lv += 1
        return lv

    def search(self, target):
        curr = self.head
        for i in range(self.level - 1, -1, -1):
            while curr.next[i] and curr.next[i].val < target:
                curr = curr.next[i]
        curr = curr.next[0]
        return curr is not None and curr.val == target

    def add(self, num):
        update = [self.head] * self.MAX_LEVEL
        curr = self.head
        for i in range(self.level - 1, -1, -1):
            while curr.next[i] and curr.next[i].val < num:
                curr = curr.next[i]
            update[i] = curr
        lv = self._random_level()
        if lv > self.level:
            for i in range(self.level, lv):
                update[i] = self.head
            self.level = lv
        new_node = self.Node(num, lv)
        for i in range(lv):
            new_node.next[i] = update[i].next[i]
            update[i].next[i] = new_node

    def erase(self, num):
        update = [None] * self.MAX_LEVEL
        curr = self.head
        for i in range(self.level - 1, -1, -1):
            while curr.next[i] and curr.next[i].val < num:
                curr = curr.next[i]
            update[i] = curr
        target = update[0].next[0]
        if not target or target.val != num:
            return False
        for i in range(self.level):
            if update[i].next[i] != target:
                break
            update[i].next[i] = target.next[i]
        return True

sl = Skiplist()
sl.add(1); sl.add(2); sl.add(3)
assert sl.search(0) == False
sl.add(4)
assert sl.search(1) == True
assert sl.erase(0) == False
assert sl.erase(1) == True
assert sl.search(1) == False
print("All tests passed!")

## Hard 7 — Reverse Nodes in Even Length Groups (LeetCode 2074)

> 🏢 **Asked by:** Amazon, Google
Reverse the nodes in each group with even length.

### Approach
Groups have lengths 1, 2, 3, 4, ... (or less for the last). For each group, count its actual length. If even, reverse it in place and reconnect.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def reverseEvenLengthGroups(head):
    prev = head
    group_len = 2
    while prev.next:
        # count actual nodes in this group
        node = prev
        cnt = 0
        for _ in range(group_len):
            if not node.next:
                break
            node = node.next
            cnt += 1
        if cnt % 2 == 1:
            prev = node
        else:
            # reverse cnt nodes starting from prev.next
            curr = prev.next
            p = None
            for _ in range(cnt):
                nxt = curr.next
                curr.next = p
                p = curr
                curr = nxt
            tail = prev.next
            prev.next = p
            tail.next = curr
            prev = tail
        group_len += 1
    return head

assert to_list(reverseEvenLengthGroups(make_list([5,2,6,3,9,1,7,3,8,4]))) == [5,6,2,3,9,1,4,8,3,7]
assert to_list(reverseEvenLengthGroups(make_list([1,1,0,6]))) == [1,0,1,6]
print("All tests passed!")

## Hard 8 — Add Two Numbers II (LeetCode 445)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Bloomberg, Apple
Add two numbers stored in forward order (most significant digit first).

### Approach
Reverse both lists, add them (like LC 2), then reverse the result. Or use stacks to avoid modifying the input.

**Time:** O(n+m) | **Space:** O(n+m)

In [ ]:
def addTwoNumbersII(l1, l2):
    s1, s2 = [], []
    while l1: s1.append(l1.val); l1 = l1.next
    while l2: s2.append(l2.val); l2 = l2.next

    carry = 0
    head = None
    while s1 or s2 or carry:
        val = carry
        if s1: val += s1.pop()
        if s2: val += s2.pop()
        carry, val = divmod(val, 10)
        node = ListNode(val)
        node.next = head
        head = node
    return head

# 7243 + 564 = 7807
assert to_list(addTwoNumbersII(make_list([7,2,4,3]), make_list([5,6,4]))) == [7,8,0,7]
assert to_list(addTwoNumbersII(make_list([5]), make_list([5]))) == [1,0]
print("All tests passed!")

## Hard 9 — All O`one Data Structure (LeetCode 432)

> 🏢 **Asked by:** Amazon, Google
Design a data structure with O(1) inc, dec, getMaxKey, getMinKey.

### Approach
Use a **doubly linked list** where each node holds a count and a set of keys with that count. A hash map links each key to its current node. This enables O(1) all operations.

**Time:** O(1) for all | **Space:** O(n)

In [ ]:
class AllOne:
    class Node:
        def __init__(self, count=0):
            self.count = count
            self.keys = set()
            self.prev = self.next = None

    def __init__(self):
        self.head = self.Node()  # min sentinel
        self.tail = self.Node()  # max sentinel
        self.head.next = self.tail
        self.tail.prev = self.head
        self.key_node = {}  # key -> Node

    def _insert_after(self, node, new_count):
        new = self.Node(new_count)
        new.prev = node
        new.next = node.next
        node.next.prev = new
        node.next = new
        return new

    def _remove(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev

    def inc(self, key):
        if key in self.key_node:
            node = self.key_node[key]
            new_count = node.count + 1
            if node.next.count != new_count:
                self._insert_after(node, new_count)
            node.next.keys.add(key)
            self.key_node[key] = node.next
            node.keys.remove(key)
            if not node.keys:
                self._remove(node)
        else:
            if self.head.next.count != 1:
                self._insert_after(self.head, 1)
            self.head.next.keys.add(key)
            self.key_node[key] = self.head.next

    def dec(self, key):
        node = self.key_node[key]
        node.keys.remove(key)
        if node.count == 1:
            del self.key_node[key]
        else:
            new_count = node.count - 1
            if node.prev.count != new_count:
                self._insert_after(node.prev, new_count)
            node.prev.keys.add(key)
            self.key_node[key] = node.prev
        if not node.keys:
            self._remove(node)

    def getMaxKey(self):
        return next(iter(self.tail.prev.keys)) if self.tail.prev != self.head else ""

    def getMinKey(self):
        return next(iter(self.head.next.keys)) if self.head.next != self.tail else ""

ao = AllOne()
ao.inc("a"); ao.inc("a"); ao.inc("b"); ao.inc("b"); ao.inc("b")
assert ao.getMaxKey() == "b"
assert ao.getMinKey() == "a"
ao.dec("b"); ao.dec("b")
assert ao.getMaxKey() == ao.getMinKey()  # both are equal count now
print("All tests passed!")

## Hard 10 — Sort List — Full O(n log n) O(1) space (LeetCode 148)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Meta
Bottom-up merge sort for O(1) space.

### Approach
Instead of recursive top-down merge sort (O(log n) stack), use **bottom-up** merge sort: start with segments of size 1, merge into size 2, then 4, then 8... This achieves O(1) space.

**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def sortListBottomUp(head):
    def merge(l1, l2):
        dummy = ListNode(0)
        curr = dummy
        while l1 and l2:
            if l1.val <= l2.val:
                curr.next = l1; l1 = l1.next
            else:
                curr.next = l2; l2 = l2.next
            curr = curr.next
        curr.next = l1 or l2
        return dummy.next

    def split(head, size):
        """Split list into two: first 'size' nodes and the rest"""
        for _ in range(size - 1):
            if not head:
                return None, None
            head = head.next
        if not head:
            return None, None
        second = head.next
        head.next = None
        return second

    n = 0
    curr = head
    while curr: n += 1; curr = curr.next

    dummy = ListNode(0, head)
    size = 1
    while size < n:
        tail = dummy
        curr = dummy.next
        while curr:
            left = curr
            right = split(left, size)
            curr = split(right, size) if right else None
            merged = merge(left, right)
            tail.next = merged
            while tail.next:
                tail = tail.next
        size *= 2
    return dummy.next

assert to_list(sortListBottomUp(make_list([4,2,1,3]))) == [1,2,3,4]
assert to_list(sortListBottomUp(make_list([-1,5,3,4,0]))) == [-1,0,3,4,5]
print("All tests passed!")

## Easy Problems (21-40)


## Easy 21 — Length of Linked List


> 🏢 **Asked by:** Amazon, Google
Count the number of nodes in a singly linked list by traversing with a loop.


### Approach
Traverse from head to None, incrementing a counter at each node.

**Time:** O(n) | **Space:** O(1)

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def make_list(vals):
    dummy = ListNode(0)
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def to_list(head):
    res = []
    while head:
        res.append(head.val)
        head = head.next
    return res

def lengthOfLinkedList(head):
    count = 0
    cur = head
    while cur:
        count += 1
        cur = cur.next
    return count

assert lengthOfLinkedList(make_list([1, 2, 3, 4, 5])) == 5
assert lengthOfLinkedList(make_list([])) == 0
assert lengthOfLinkedList(make_list([7])) == 1
print("All tests passed!")


## Easy 22 — Check if Linked List is Sorted


> 🏢 **Asked by:** Amazon, Google
Return True if the linked list values are in non-decreasing order.


### Approach
Walk node by node; if any node's value exceeds its successor's, return False.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def isSortedLinkedList(head):
    cur = head
    while cur and cur.next:
        if cur.val > cur.next.val:
            return False
        cur = cur.next
    return True

assert isSortedLinkedList(make_list([1, 2, 3, 4])) == True
assert isSortedLinkedList(make_list([1, 3, 2, 4])) == False
assert isSortedLinkedList(make_list([5])) == True
assert isSortedLinkedList(make_list([])) == True
print("All tests passed!")


## Easy 23 — Sum of All Nodes in Linked List


> 🏢 **Asked by:** Amazon, Google
Return the sum of all node values in the linked list.


### Approach
Single pass, accumulate values.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def sumLinkedList(head):
    total = 0
    cur = head
    while cur:
        total += cur.val
        cur = cur.next
    return total

assert sumLinkedList(make_list([1, 2, 3, 4, 5])) == 15
assert sumLinkedList(make_list([])) == 0
assert sumLinkedList(make_list([-1, 2, -3])) == -2
print("All tests passed!")


## Easy 24 — Find Maximum Value in Linked List


> 🏢 **Asked by:** Amazon, Google
Return the maximum node value in a non-empty linked list.


### Approach
Track a running maximum as you traverse.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def maxValueLinkedList(head):
    max_val = head.val
    cur = head.next
    while cur:
        if cur.val > max_val:
            max_val = cur.val
        cur = cur.next
    return max_val

assert maxValueLinkedList(make_list([3, 1, 4, 1, 5, 9, 2])) == 9
assert maxValueLinkedList(make_list([-5, -1, -3])) == -1
assert maxValueLinkedList(make_list([42])) == 42
print("All tests passed!")


## Easy 25 — Search for Value in Linked List


> 🏢 **Asked by:** Amazon, Google
Return the 0-based index of the first occurrence of `target`, or -1 if not found.


### Approach
Linear scan with an index counter.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def searchLinkedList(head, target):
    idx = 0
    cur = head
    while cur:
        if cur.val == target:
            return idx
        cur = cur.next
        idx += 1
    return -1

assert searchLinkedList(make_list([1, 2, 3, 4]), 3) == 2
assert searchLinkedList(make_list([1, 2, 3, 4]), 5) == -1
assert searchLinkedList(make_list([7]), 7) == 0
print("All tests passed!")


## Easy 26 — Check if Value Exists in Linked List


> 🏢 **Asked by:** Amazon, Google
Return True if `target` exists anywhere in the linked list.


### Approach
Short-circuit linear scan.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def existsInLinkedList(head, target):
    cur = head
    while cur:
        if cur.val == target:
            return True
        cur = cur.next
    return False

assert existsInLinkedList(make_list([1, 2, 3]), 2) == True
assert existsInLinkedList(make_list([1, 2, 3]), 9) == False
assert existsInLinkedList(make_list([]), 1) == False
print("All tests passed!")


## Easy 27 — Append Node to End of Linked List


> 🏢 **Asked by:** Amazon, Google
Append a new node with the given value at the tail of the list.


### Approach
Walk to the last node, then attach a new node. Handle empty list by returning the new node as head.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def appendNode(head, val):
    new_node = ListNode(val)
    if not head:
        return new_node
    cur = head
    while cur.next:
        cur = cur.next
    cur.next = new_node
    return head

assert to_list(appendNode(make_list([1, 2, 3]), 4)) == [1, 2, 3, 4]
assert to_list(appendNode(make_list([]), 5)) == [5]
print("All tests passed!")


## Easy 28 — Insert Node at Beginning of Linked List


> 🏢 **Asked by:** Amazon, Google
Prepend a new node with the given value at the head.


### Approach
Create a new node whose `next` points to the current head, then return the new node as the new head.

**Time:** O(1) | **Space:** O(1)

In [ ]:
def prependNode(head, val):
    return ListNode(val, head)

assert to_list(prependNode(make_list([2, 3, 4]), 1)) == [1, 2, 3, 4]
assert to_list(prependNode(make_list([]), 7)) == [7]
print("All tests passed!")


## Easy 29 — Get Node at Index k from Linked List


> 🏢 **Asked by:** Amazon, Google
Return the value at 0-based index `k`, or -1 if out of bounds.


### Approach
Walk `k` steps from the head; return the node's value if reachable.

**Time:** O(k) | **Space:** O(1)

In [ ]:
def getNodeAtIndex(head, k):
    cur = head
    for _ in range(k):
        if not cur:
            return -1
        cur = cur.next
    return cur.val if cur else -1

assert getNodeAtIndex(make_list([10, 20, 30, 40]), 2) == 30
assert getNodeAtIndex(make_list([10, 20, 30, 40]), 0) == 10
assert getNodeAtIndex(make_list([10, 20, 30, 40]), 5) == -1
print("All tests passed!")


## Easy 30 — Remove Node at Index k from Linked List


> 🏢 **Asked by:** Amazon, Google
Remove the node at 0-based index `k` and return the new head.


### Approach
Use a dummy node. Walk to the node just before index `k` and re-link to skip it.

**Time:** O(k) | **Space:** O(1)

In [ ]:
def removeAtIndex(head, k):
    dummy = ListNode(0, head)
    prev = dummy
    for _ in range(k):
        if not prev.next:
            return dummy.next
        prev = prev.next
    if prev.next:
        prev.next = prev.next.next
    return dummy.next

assert to_list(removeAtIndex(make_list([1, 2, 3, 4, 5]), 2)) == [1, 2, 4, 5]
assert to_list(removeAtIndex(make_list([1, 2, 3]), 0)) == [2, 3]
assert to_list(removeAtIndex(make_list([1, 2, 3]), 2)) == [1, 2]
print("All tests passed!")


## Easy 31 — Print Linked List in Reverse


> 🏢 **Asked by:** Amazon, Google
Collect all values to an array and return them reversed, without modifying the list.


### Approach
Traverse and collect node values into a list, then reverse the list in place.

**Time:** O(n) | **Space:** O(n)

In [ ]:
def linkedListReversed(head):
    vals = []
    cur = head
    while cur:
        vals.append(cur.val)
        cur = cur.next
    return vals[::-1]

assert linkedListReversed(make_list([1, 2, 3, 4])) == [4, 3, 2, 1]
assert linkedListReversed(make_list([5])) == [5]
assert linkedListReversed(make_list([])) == []
print("All tests passed!")


## Easy 32 — Duplicate the Linked List (Shallow Copy)


> 🏢 **Asked by:** Amazon, Google
Create a new linked list with the same values as the original.


### Approach
Traverse the original list, creating a new node for each value. Use a dummy head to simplify tail tracking.

**Time:** O(n) | **Space:** O(n)

In [ ]:
def duplicateLinkedList(head):
    dummy = ListNode(0)
    tail = dummy
    cur = head
    while cur:
        tail.next = ListNode(cur.val)
        tail = tail.next
        cur = cur.next
    return dummy.next

original = make_list([1, 2, 3, 4])
copy = duplicateLinkedList(original)
assert to_list(copy) == [1, 2, 3, 4]
assert copy is not original
print("All tests passed!")


## Easy 33 — Compare Two Linked Lists


> 🏢 **Asked by:** Amazon, Google
Return True if two linked lists have the same values in the same order.


### Approach
Walk both lists simultaneously; if values ever differ or lengths differ, return False.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def compareLinkedLists(h1, h2):
    a, b = h1, h2
    while a and b:
        if a.val != b.val:
            return False
        a, b = a.next, b.next
    return a is None and b is None

assert compareLinkedLists(make_list([1, 2, 3]), make_list([1, 2, 3])) == True
assert compareLinkedLists(make_list([1, 2, 3]), make_list([1, 2, 4])) == False
assert compareLinkedLists(make_list([1, 2]), make_list([1, 2, 3])) == False
assert compareLinkedLists(make_list([]), make_list([])) == True
print("All tests passed!")


## Easy 34 — Merge Two Sorted Linked Lists (Recursive)


> 🏢 **Asked by:** Amazon, Google, Microsoft, Apple, Bloomberg
Merge two sorted linked lists into one sorted list using recursion.


### Approach
Base case: if either list is empty return the other. Recursively link the smaller head to the result of merging the rest.

**Time:** O(n + m) | **Space:** O(n + m) call stack

In [ ]:
def mergeTwoSortedRecursive(l1, l2):
    if not l1: return l2
    if not l2: return l1
    if l1.val <= l2.val:
        l1.next = mergeTwoSortedRecursive(l1.next, l2)
        return l1
    else:
        l2.next = mergeTwoSortedRecursive(l1, l2.next)
        return l2

assert to_list(mergeTwoSortedRecursive(make_list([1, 3, 5]), make_list([2, 4, 6]))) == [1, 2, 3, 4, 5, 6]
assert to_list(mergeTwoSortedRecursive(make_list([1, 2, 3]), make_list([]))) == [1, 2, 3]
assert to_list(mergeTwoSortedRecursive(make_list([]), make_list([1]))) == [1]
print("All tests passed!")


## Easy 35 — Check if Linked List has Even Length


> 🏢 **Asked by:** Amazon, Google
Return True if the list contains an even number of nodes.


### Approach
Advance a single pointer two steps at a time. If it lands exactly on None, length is even.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def isEvenLength(head):
    cur = head
    while cur and cur.next:
        cur = cur.next.next
    return cur is None

assert isEvenLength(make_list([1, 2, 3, 4])) == True
assert isEvenLength(make_list([1, 2, 3])) == False
assert isEvenLength(make_list([])) == True
assert isEvenLength(make_list([1])) == False
print("All tests passed!")


## Easy 36 — Rotate Linked List Left by k


> 🏢 **Asked by:** Amazon, Google
Rotate the linked list **left** by `k` positions (opposite of LC 61's right rotation).


### Approach
Find length `n`, then `k = k % n`. Walk to the new tail (index `k-1`), set its next as new head, and close the loop.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def rotateLeft(head, k):
    if not head or not head.next or k == 0:
        return head
    n = 1
    tail = head
    while tail.next:
        tail = tail.next
        n += 1
    k = k % n
    if k == 0:
        return head
    cur = head
    for _ in range(k - 1):
        cur = cur.next
    new_head = cur.next
    cur.next = None
    tail.next = head
    return new_head

assert to_list(rotateLeft(make_list([1, 2, 3, 4, 5]), 2)) == [3, 4, 5, 1, 2]
assert to_list(rotateLeft(make_list([0, 1, 2]), 4)) == [1, 2, 0]
print("All tests passed!")


## Easy 37 — Find Second to Last Node in Linked List


> 🏢 **Asked by:** Amazon, Google
Return the value of the second-to-last node, or -1 if the list has fewer than 2 nodes.


### Approach
Walk until `cur.next.next` is None; at that point `cur` is the second-to-last node.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def secondToLast(head):
    if not head or not head.next:
        return -1
    cur = head
    while cur.next.next:
        cur = cur.next
    return cur.val

assert secondToLast(make_list([1, 2, 3, 4])) == 3
assert secondToLast(make_list([1, 2])) == 1
assert secondToLast(make_list([1])) == -1
assert secondToLast(make_list([])) == -1
print("All tests passed!")


## Easy 38 — Zip Two Linked Lists (Interleave)


> 🏢 **Asked by:** Amazon, Google
Interleave nodes from two lists: l1[0] -> l2[0] -> l1[1] -> l2[1] -> ... Remaining nodes are appended.


### Approach
Alternate taking one node from each list. When one list runs out, attach the rest of the other.

**Time:** O(n + m) | **Space:** O(1)

In [ ]:
def zipLinkedLists(l1, l2):
    dummy = ListNode(0)
    tail = dummy
    toggle = True
    while l1 and l2:
        if toggle:
            tail.next = l1
            l1 = l1.next
        else:
            tail.next = l2
            l2 = l2.next
        tail = tail.next
        toggle = not toggle
    tail.next = l1 if l1 else l2
    return dummy.next

assert to_list(zipLinkedLists(make_list([1, 3, 5]), make_list([2, 4, 6]))) == [1, 2, 3, 4, 5, 6]
assert to_list(zipLinkedLists(make_list([1, 2]), make_list([3, 4, 5, 6]))) == [1, 3, 2, 4, 5, 6]
print("All tests passed!")


## Easy 39 — Check if Linked List is a Palindrome Using Stack


> 🏢 **Asked by:** Amazon, Google
Determine if the linked list reads the same forwards and backwards, using a stack.


### Approach
Push all values onto a stack (LIFO), then compare stack pops against a second traversal from the head.

**Time:** O(n) | **Space:** O(n)

In [ ]:
def isPalindromeStack(head):
    stack = []
    cur = head
    while cur:
        stack.append(cur.val)
        cur = cur.next
    cur = head
    while cur:
        if cur.val != stack.pop():
            return False
        cur = cur.next
    return True

assert isPalindromeStack(make_list([1, 2, 2, 1])) == True
assert isPalindromeStack(make_list([1, 2, 3, 2, 1])) == True
assert isPalindromeStack(make_list([1, 2, 3])) == False
assert isPalindromeStack(make_list([1])) == True
print("All tests passed!")


## Easy 40 — Remove All Occurrences of Value (Iterative)


> 🏢 **Asked by:** Amazon, Google
Remove every node whose value equals `val` and return the updated head. (LC 203 iterative approach)


### Approach
Use a dummy node before head. Traverse with `prev` and `cur`; skip nodes that match `val`, otherwise advance `prev`.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def removeAllOccurrences(head, val):
    dummy = ListNode(0, head)
    prev, cur = dummy, head
    while cur:
        if cur.val == val:
            prev.next = cur.next
        else:
            prev = cur
        cur = cur.next
    return dummy.next

assert to_list(removeAllOccurrences(make_list([1, 2, 6, 3, 4, 5, 6]), 6)) == [1, 2, 3, 4, 5]
assert to_list(removeAllOccurrences(make_list([7, 7, 7, 7]), 7)) == []
assert to_list(removeAllOccurrences(make_list([1, 2, 3]), 4)) == [1, 2, 3]
print("All tests passed!")


## Medium Problems (16-30)


## Medium 16 — Design Linked List (LC 707)


> 🏢 **Asked by:** Amazon, Google
Implement a singly linked list supporting: `get(index)`, `addAtHead(val)`, `addAtTail(val)`, `addAtIndex(index, val)`, `deleteAtIndex(index)`.


### Approach
Maintain `head`, `tail`, and `size`. Use a dummy head to simplify edge cases for insert/delete.

**Time:** O(n) per operation | **Space:** O(n)

In [ ]:
class MyLinkedList:
    def __init__(self):
        self.dummy = ListNode(0)
        self.size = 0

    def get(self, index):
        if index < 0 or index >= self.size:
            return -1
        cur = self.dummy.next
        for _ in range(index):
            cur = cur.next
        return cur.val

    def addAtHead(self, val):
        self.addAtIndex(0, val)

    def addAtTail(self, val):
        self.addAtIndex(self.size, val)

    def addAtIndex(self, index, val):
        if index < 0 or index > self.size:
            return
        prev = self.dummy
        for _ in range(index):
            prev = prev.next
        prev.next = ListNode(val, prev.next)
        self.size += 1

    def deleteAtIndex(self, index):
        if index < 0 or index >= self.size:
            return
        prev = self.dummy
        for _ in range(index):
            prev = prev.next
        prev.next = prev.next.next
        self.size -= 1

ll = MyLinkedList()
ll.addAtHead(1)
ll.addAtTail(3)
ll.addAtIndex(1, 2)
assert ll.get(0) == 1
assert ll.get(1) == 2
assert ll.get(2) == 3
ll.deleteAtIndex(1)
assert ll.get(1) == 3
print("All tests passed!")


## Medium 17 — Double a Number Represented as a Linked List (LC 2816)


> 🏢 **Asked by:** Amazon, Google
Given a linked list representing a non-negative integer (MSB first), double it and return the result as a linked list.


### Approach
Reverse the list to process least-significant digit first. Multiply each digit by 2 with carry. Reverse back.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def doubleLinkedList(head):
    def reverse(node):
        prev = None
        while node:
            nxt = node.next
            node.next = prev
            prev = node
            node = nxt
        return prev

    head = reverse(head)
    carry = 0
    cur = head
    prev = None
    while cur:
        val = cur.val * 2 + carry
        cur.val = val % 10
        carry = val // 10
        prev = cur
        cur = cur.next
    if carry:
        prev.next = ListNode(carry)
    return reverse(head)

assert to_list(doubleLinkedList(make_list([1, 8, 9]))) == [3, 7, 8]
assert to_list(doubleLinkedList(make_list([9, 9, 9]))) == [1, 9, 9, 8]
assert to_list(doubleLinkedList(make_list([0]))) == [0]
print("All tests passed!")


## Medium 18 — Min and Max Number of Nodes Between Critical Points (LC 2058)


> 🏢 **Asked by:** Amazon, Google
A *critical point* is a local minimum or maximum. Return `[minDist, maxDist]` between any two critical points, or `[-1, -1]` if fewer than two exist.


### Approach
Walk the list, tracking critical point positions. Max distance is always `last - first`. Min distance is the minimum of consecutive critical point gaps.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def nodesBetweenCriticalPoints(head):
    prev, cur = head, head.next
    idx = 1
    first_cp = last_cp = None
    prev_cp = None
    min_dist = float('inf')

    while cur and cur.next:
        if (cur.val > prev.val and cur.val > cur.next.val) or \
           (cur.val < prev.val and cur.val < cur.next.val):
            if prev_cp is not None:
                min_dist = min(min_dist, idx - prev_cp)
            if first_cp is None:
                first_cp = idx
            last_cp = idx
            prev_cp = idx
        prev, cur = cur, cur.next
        idx += 1

    if first_cp == last_cp:
        return [-1, -1]
    return [min_dist, last_cp - first_cp]

assert nodesBetweenCriticalPoints(make_list([3, 1, 2, 2, 3, 2, 2, 2, 7])) == [3, 3]
assert nodesBetweenCriticalPoints(make_list([5, 3, 1, 2, 5, 1, 2])) == [1, 3]
print("All tests passed!")


## Medium 19 — Find All Twin Sums in Linked List


> 🏢 **Asked by:** Amazon, Google
For an even-length linked list, twin `i` and twin `n-1-i` are paired. Return a list of all twin sums.


### Approach
Collect all values, then pair `vals[i] + vals[n-1-i]` for `i` in `0..n//2-1`.

**Time:** O(n) | **Space:** O(n)

In [ ]:
def allTwinSums(head):
    vals = []
    cur = head
    while cur:
        vals.append(cur.val)
        cur = cur.next
    n = len(vals)
    return [vals[i] + vals[n - 1 - i] for i in range(n // 2)]

assert allTwinSums(make_list([5, 4, 2, 1])) == [6, 6]
assert allTwinSums(make_list([1, 2, 3, 4])) == [5, 5]
assert allTwinSums(make_list([1, 100000])) == [100001]
print("All tests passed!")


## Medium 20 — Convert Sorted List to Binary Search Tree (LC 109)


> 🏢 **Asked by:** Amazon, Google
Given a sorted linked list, convert it to a height-balanced BST.


### Approach
Use fast/slow pointers to find the middle (root). Recursively build left subtree from the left half and right subtree from the right half.

**Time:** O(n log n) | **Space:** O(log n) call stack

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def sortedListToBST(head):
    if not head:
        return None
    if not head.next:
        return TreeNode(head.val)
    slow, fast, prev = head, head, None
    while fast and fast.next:
        prev = slow
        slow = slow.next
        fast = fast.next.next
    prev.next = None
    root = TreeNode(slow.val)
    root.left = sortedListToBST(head)
    root.right = sortedListToBST(slow.next)
    return root

def inorder(root):
    if not root: return []
    return inorder(root.left) + [root.val] + inorder(root.right)

bst = sortedListToBST(make_list([-10, -3, 0, 5, 9]))
assert inorder(bst) == [-10, -3, 0, 5, 9]
bst2 = sortedListToBST(make_list([1, 3]))
assert inorder(bst2) == [1, 3]
print("All tests passed!")


## Medium 21 — Flatten a Multilevel Doubly Linked List (LC 430)


> 🏢 **Asked by:** Amazon, Google, Microsoft
A doubly linked list may have a `child` pointer to another doubly linked list. Flatten it so all nodes appear in a single-level list.


### Approach
When we encounter a node with a child, splice the child list between the current node and its next. Use a stack to handle nested children.

**Time:** O(n) | **Space:** O(depth) stack

In [ ]:
class DLNode:
    def __init__(self, val=0, prev=None, next=None, child=None):
        self.val = val
        self.prev = prev
        self.next = next
        self.child = child

def flattenDLL(head):
    if not head:
        return head
    stack = []
    cur = head
    while cur:
        if cur.child:
            if cur.next:
                stack.append(cur.next)
            cur.next = cur.child
            cur.child.prev = cur
            cur.child = None
        if not cur.next and stack:
            nxt = stack.pop()
            cur.next = nxt
            nxt.prev = cur
        cur = cur.next
    return head

n1, n2, n3 = DLNode(1), DLNode(2), DLNode(3)
n4, n5 = DLNode(4), DLNode(5)
n1.next = n2; n2.prev = n1; n2.next = n3; n3.prev = n2
n2.child = n4; n4.next = n5; n5.prev = n4
result = flattenDLL(n1)
vals = []
cur = result
while cur:
    vals.append(cur.val)
    cur = cur.next
assert vals == [1, 2, 4, 5, 3]
print("All tests passed!")


## Medium 22 — Insert Greatest Common Divisors in Linked List (LC 2807)


> 🏢 **Asked by:** Amazon, Google
Between every two adjacent nodes, insert a new node with their GCD value.


### Approach
Traverse the list; between each consecutive pair, create and splice a GCD node.

**Time:** O(n log M) where M is max value | **Space:** O(1)

In [ ]:
from math import gcd

def insertGCDs(head):
    cur = head
    while cur and cur.next:
        g = gcd(cur.val, cur.next.val)
        gcd_node = ListNode(g, cur.next)
        cur.next = gcd_node
        cur = gcd_node.next
    return head

assert to_list(insertGCDs(make_list([18, 6, 10, 3]))) == [18, 6, 6, 2, 10, 1, 3]
assert to_list(insertGCDs(make_list([7]))) == [7]
assert to_list(insertGCDs(make_list([4, 8]))) == [4, 4, 8]
print("All tests passed!")


## Medium 23 — Delete the Middle Node with Minimal Passes


> 🏢 **Asked by:** Amazon, Google
Delete the middle node of a linked list in a single pass using fast/slow pointers.


### Approach
Advance `fast` two steps and `slow` one step. Track `prev` of `slow`. When `fast` reaches the end, `slow` is at the middle; unlink it via `prev`.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def deleteMiddleNode(head):
    if not head or not head.next:
        return None
    slow, fast, prev = head, head, None
    while fast and fast.next:
        prev = slow
        slow = slow.next
        fast = fast.next.next
    prev.next = slow.next
    return head

assert to_list(deleteMiddleNode(make_list([1, 3, 4, 7, 1, 2, 6]))) == [1, 3, 4, 1, 2, 6]
assert to_list(deleteMiddleNode(make_list([1, 2, 3, 4]))) == [1, 2, 4]
assert to_list(deleteMiddleNode(make_list([2, 1]))) == [2]
print("All tests passed!")


## Medium 24 — Merge Nodes in Between Zeros (LC 2181)


> 🏢 **Asked by:** Amazon, Google
Given a linked list where values between consecutive 0s should be summed, return a new list of those sums.


### Approach
Skip the initial zero. Accumulate the sum until the next zero; emit a node with that sum, then reset.

**Time:** O(n) | **Space:** O(1) output excluded

In [ ]:
def mergeNodes(head):
    dummy = ListNode(0)
    tail = dummy
    cur = head.next
    current_sum = 0
    while cur:
        if cur.val == 0:
            tail.next = ListNode(current_sum)
            tail = tail.next
            current_sum = 0
        else:
            current_sum += cur.val
        cur = cur.next
    return dummy.next

assert to_list(mergeNodes(make_list([0, 3, 1, 0, 4, 5, 2, 0]))) == [4, 11]
assert to_list(mergeNodes(make_list([0, 1, 0, 3, 0, 2, 2, 0]))) == [1, 3, 4]
print("All tests passed!")


## Medium 25 — Remove Nodes From Linked List (LC 2487)


> 🏢 **Asked by:** Amazon, Google
Remove every node that has a node with a strictly greater value to its right.


### Approach
Reverse the list, then greedily keep only nodes with non-decreasing values. Reverse back.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def removeNodesLL(head):
    def reverse(h):
        prev = None
        while h:
            h.next, prev, h = prev, h, h.next
        return prev

    head = reverse(head)
    cur = head
    max_val = head.val
    while cur and cur.next:
        if cur.next.val < max_val:
            cur.next = cur.next.next
        else:
            max_val = cur.next.val
            cur = cur.next
    return reverse(head)

assert to_list(removeNodesLL(make_list([5, 2, 13, 3, 8]))) == [13, 8]
assert to_list(removeNodesLL(make_list([1, 1, 1, 1]))) == [1, 1, 1, 1]
print("All tests passed!")


## Medium 26 — Double a Number Represented as Linked List — Stack Variant (LC 2816)


> 🏢 **Asked by:** Amazon, Google
Alternative approach using a stack to process digits without reversing in place.


### Approach
Push all digits onto a stack. Pop and double each digit applying carry. Build result list by prepending nodes.

**Time:** O(n) | **Space:** O(n)

In [ ]:
def doubleLinkedListStack(head):
    stack = []
    cur = head
    while cur:
        stack.append(cur.val)
        cur = cur.next
    carry = 0
    result_head = None
    while stack or carry:
        digit = stack.pop() if stack else 0
        val = digit * 2 + carry
        carry = val // 10
        result_head = ListNode(val % 10, result_head)
    return result_head

assert to_list(doubleLinkedListStack(make_list([1, 8, 9]))) == [3, 7, 8]
assert to_list(doubleLinkedListStack(make_list([9, 9, 9]))) == [1, 9, 9, 8]
print("All tests passed!")


## Medium 27 — Split Linked List in Parts — Fill Largest First (LC 725 Variant)


> 🏢 **Asked by:** Amazon, Google
Given a linked list and integer `k`, split into `k` parts. Larger parts come first; no part differs in size by more than 1.


### Approach
Compute `n` and `k`. Each part has size `base = n // k`; the first `n % k` parts get one extra node. Cut the list accordingly.

**Time:** O(n) | **Space:** O(k)

In [ ]:
def splitListInParts(head, k):
    n = 0
    cur = head
    while cur:
        n += 1
        cur = cur.next
    base, extra = n // k, n % k
    parts = []
    cur = head
    for i in range(k):
        part_head = cur
        part_size = base + (1 if i < extra else 0)
        for _ in range(part_size - 1):
            if cur:
                cur = cur.next
        if cur:
            cur.next, cur = None, cur.next
        parts.append(part_head)
    return parts

parts = splitListInParts(make_list([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]), 3)
assert [to_list(p) for p in parts] == [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]
parts2 = splitListInParts(make_list([1, 2, 3]), 5)
assert [to_list(p) for p in parts2] == [[1], [2], [3], [], []]
print("All tests passed!")


## Medium 28 — Add One to a Number Represented as Linked List


> 🏢 **Asked by:** Amazon, Google
The linked list stores digits MSB to LSB. Add 1 and return the resulting linked list.


### Approach
Reverse the list to process LSB first, add 1 with carry propagation, then reverse back.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def addOneToLinkedList(head):
    def reverse(h):
        prev = None
        while h:
            h.next, prev, h = prev, h, h.next
        return prev

    head = reverse(head)
    carry = 1
    cur = head
    prev = None
    while cur and carry:
        val = cur.val + carry
        cur.val = val % 10
        carry = val // 10
        prev = cur
        cur = cur.next
    if carry:
        prev.next = ListNode(carry)
    return reverse(head)

assert to_list(addOneToLinkedList(make_list([1, 2, 3]))) == [1, 2, 4]
assert to_list(addOneToLinkedList(make_list([9, 9, 9]))) == [1, 0, 0, 0]
assert to_list(addOneToLinkedList(make_list([0]))) == [1]
print("All tests passed!")


## Medium 29 — Find Length of Loop in Linked List


> 🏢 **Asked by:** Amazon, Google
Given a linked list that may contain a cycle, return the length of the cycle (0 if none).


### Approach
Detect the meeting point with Floyd's algorithm. Then fix `slow` and advance `fast` one step at a time, counting steps until they meet again.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def cycleLength(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast:
            length = 1
            cur = slow.next
            while cur != slow:
                cur = cur.next
                length += 1
            return length
    return 0

nodes = [ListNode(i) for i in range(1, 6)]
for i in range(4):
    nodes[i].next = nodes[i + 1]
nodes[4].next = nodes[2]
assert cycleLength(nodes[0]) == 3
no_cycle = make_list([1, 2, 3, 4])
assert cycleLength(no_cycle) == 0
print("All tests passed!")


## Medium 30 — Count Nodes in Complete Binary Tree


> 🏢 **Asked by:** Amazon, Google
Count the number of nodes in a complete binary tree in O(log^2 n) time by leveraging the completeness property.


### Approach
Measure left height and right height. If equal, the left subtree is a perfect binary tree; recurse on the right only. Otherwise recurse on the left. This gives O(log^2 n).

**Time:** O(log^2 n) | **Space:** O(log n)

In [ ]:
def countNodes(root):
    if not root:
        return 0
    left_h = right_h = 0
    l, r = root, root
    while l:
        left_h += 1
        l = l.left
    while r:
        right_h += 1
        r = r.right
    if left_h == right_h:
        return (1 << left_h) - 1
    return 1 + countNodes(root.left) + countNodes(root.right)

nodes_bt = [TreeNode(i) for i in range(1, 7)]
for i in range(3):
    if 2 * i + 1 < 6:
        nodes_bt[i].left = nodes_bt[2 * i + 1]
    if 2 * i + 2 < 6:
        nodes_bt[i].right = nodes_bt[2 * i + 2]
assert countNodes(nodes_bt[0]) == 6
assert countNodes(None) == 0
assert countNodes(TreeNode(1)) == 1
print("All tests passed!")


## Hard Problems (11-20)


## Hard 11 — LFU Cache (LC 460)


> 🏢 **Asked by:** Amazon, Google, Meta, Bloomberg
Design a data structure for a Least Frequently Used cache with `get(key)` and `put(key, value)` operations. When capacity is reached, evict the least frequently used key (LRU tie-breaking).


### Approach
Maintain `key_val`, `key_freq` dicts, a `freq_keys` dict of OrderedDicts, and `min_freq`. On `get`/`put`, increment frequency and move the key to the correct frequency bucket.

**Time:** O(1) per operation | **Space:** O(capacity)

In [ ]:
from collections import defaultdict, OrderedDict

class LFUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.key_val = {}
        self.key_freq = {}
        self.freq_keys = defaultdict(OrderedDict)
        self.min_freq = 0

    def _update(self, key):
        freq = self.key_freq[key]
        del self.freq_keys[freq][key]
        if not self.freq_keys[freq] and freq == self.min_freq:
            self.min_freq += 1
        self.key_freq[key] = freq + 1
        self.freq_keys[freq + 1][key] = None

    def get(self, key):
        if key not in self.key_val:
            return -1
        self._update(key)
        return self.key_val[key]

    def put(self, key, value):
        if self.capacity <= 0:
            return
        if key in self.key_val:
            self.key_val[key] = value
            self._update(key)
        else:
            if len(self.key_val) >= self.capacity:
                evict_key, _ = self.freq_keys[self.min_freq].popitem(last=False)
                del self.key_val[evict_key]
                del self.key_freq[evict_key]
            self.key_val[key] = value
            self.key_freq[key] = 1
            self.freq_keys[1][key] = None
            self.min_freq = 1

lfu = LFUCache(2)
lfu.put(1, 1)
lfu.put(2, 2)
assert lfu.get(1) == 1
lfu.put(3, 3)
assert lfu.get(2) == -1
assert lfu.get(3) == 3
lfu.put(4, 4)
assert lfu.get(1) == -1
assert lfu.get(3) == 3
assert lfu.get(4) == 4
print("All tests passed!")


## Hard 12 — Reverse Linked List in Groups of K (LC 25)


> 🏢 **Asked by:** Amazon, Google, Microsoft, Meta, Apple
Reverse every consecutive group of `k` nodes. If the last group has fewer than `k` nodes, leave them as-is.


### Approach
Iteratively count `k` nodes ahead; if fewer than `k` remain, stop. Otherwise reverse the group in place, reconnect to the previous tail and next group.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def reverseKGroup(head, k):
    def has_k_nodes(node, k):
        while node and k > 0:
            node = node.next
            k -= 1
        return k == 0

    dummy = ListNode(0, head)
    group_prev = dummy
    while has_k_nodes(group_prev.next, k):
        group_start = group_prev.next
        prev, cur = None, group_start
        for _ in range(k):
            cur.next, prev, cur = prev, cur, cur.next
        group_prev.next = prev
        group_start.next = cur
        group_prev = group_start
    return dummy.next

assert to_list(reverseKGroup(make_list([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
assert to_list(reverseKGroup(make_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]
assert to_list(reverseKGroup(make_list([1]), 1)) == [1]
print("All tests passed!")


## Hard 13 — Merge K Sorted Lists — Divide and Conquer (LC 23)


> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Merge `k` sorted linked lists into one sorted list using divide and conquer.


### Approach
Repeatedly halve the list of lists and merge pairs. This gives O(n log k) vs the naive O(nk).

**Time:** O(n log k) | **Space:** O(log k) call stack

In [ ]:
def mergeKSortedDivide(lists):
    def merge2(l1, l2):
        dummy = ListNode(0)
        tail = dummy
        while l1 and l2:
            if l1.val <= l2.val:
                tail.next = l1
                l1 = l1.next
            else:
                tail.next = l2
                l2 = l2.next
            tail = tail.next
        tail.next = l1 if l1 else l2
        return dummy.next

    if not lists:
        return None
    while len(lists) > 1:
        merged = []
        for i in range(0, len(lists), 2):
            l1 = lists[i]
            l2 = lists[i + 1] if i + 1 < len(lists) else None
            merged.append(merge2(l1, l2))
        lists = merged
    return lists[0]

ls = [make_list([1, 4, 5]), make_list([1, 3, 4]), make_list([2, 6])]
assert to_list(mergeKSortedDivide(ls)) == [1, 1, 2, 3, 4, 4, 5, 6]
assert mergeKSortedDivide([]) is None
assert to_list(mergeKSortedDivide([make_list([])])) == []
print("All tests passed!")


## Hard 14 — Design Twitter (LC 355)


> 🏢 **Asked by:** Amazon, Google
Design a simplified Twitter: `postTweet`, `getNewsFeed` (10 most recent from user + followees), `follow`, `unfollow`.


### Approach
Store tweets per user as a list of `(neg_timestamp, tweetId)`. `getNewsFeed` uses a max-heap across all relevant user tweet lists.

**Time:** O(N log N) getNewsFeed, O(1) others | **Space:** O(tweets + follows)

In [ ]:
import heapq

class Twitter:
    def __init__(self):
        self.time = 0
        self.tweets = defaultdict(list)
        self.following = defaultdict(set)

    def postTweet(self, userId, tweetId):
        self.tweets[userId].append((-self.time, tweetId))
        self.time += 1

    def getNewsFeed(self, userId):
        heap = []
        sources = self.following[userId] | {userId}
        for uid in sources:
            if self.tweets[uid]:
                idx = len(self.tweets[uid]) - 1
                neg_t, tid = self.tweets[uid][idx]
                heapq.heappush(heap, (neg_t, tid, uid, idx))
        res = []
        while heap and len(res) < 10:
            neg_t, tid, uid, idx = heapq.heappop(heap)
            res.append(tid)
            if idx > 0:
                idx -= 1
                neg_t2, tid2 = self.tweets[uid][idx]
                heapq.heappush(heap, (neg_t2, tid2, uid, idx))
        return res

    def follow(self, followerId, followeeId):
        self.following[followerId].add(followeeId)

    def unfollow(self, followerId, followeeId):
        self.following[followerId].discard(followeeId)

tw = Twitter()
tw.postTweet(1, 5)
assert tw.getNewsFeed(1) == [5]
tw.follow(1, 2)
tw.postTweet(2, 6)
assert tw.getNewsFeed(1) == [6, 5]
tw.unfollow(1, 2)
assert tw.getNewsFeed(1) == [5]
print("All tests passed!")


## Hard 15 — Text Justification (LC 68)


> 🏢 **Asked by:** Amazon, Google
Given a list of words and a max line width `maxWidth`, format the text so each line has exactly `maxWidth` characters with full justification.


### Approach
Greedily pack words onto each line. For non-last lines, distribute spaces evenly (extras go left). The last line is left-justified.

**Time:** O(n * maxWidth) | **Space:** O(n)

In [ ]:
def fullJustify(words, maxWidth):
    lines, cur_line, cur_len = [], [], 0
    for w in words:
        if cur_len + len(w) + len(cur_line) > maxWidth:
            lines.append(cur_line)
            cur_line, cur_len = [], 0
        cur_line.append(w)
        cur_len += len(w)
    lines.append(cur_line)

    result = []
    for i, line in enumerate(lines):
        if i == len(lines) - 1 or len(line) == 1:
            result.append(' '.join(line).ljust(maxWidth))
        else:
            total_spaces = maxWidth - sum(len(w) for w in line)
            gaps = len(line) - 1
            space, extra = divmod(total_spaces, gaps)
            row = line[0]
            for j in range(1, len(line)):
                row += ' ' * (space + (1 if j <= extra else 0)) + line[j]
            result.append(row)
    return result

out = fullJustify(["This", "is", "an", "example", "of", "text", "justification."], 16)
assert out == ["This    is    an", "example  of text", "justification.  "]
out2 = fullJustify(["What", "must", "be", "acknowledgment", "shall", "be"], 16)
assert out2 == ["What   must   be", "acknowledgment  ", "shall be        "]
print("All tests passed!")


## Hard 16 — Serialize and Deserialize Linked List with Random Pointers


> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Serialize a linked list (with `next` and `random` pointers) to a string, then deserialize back to the original structure.


### Approach
Serialize: assign indices to nodes, encode each as `(val, random_index_or_-1)`. Deserialize: create nodes, then re-link `next` and `random` using the index map.

**Time:** O(n) | **Space:** O(n)

In [ ]:
class RandomNode:
    def __init__(self, val=0, next=None, random=None):
        self.val = val
        self.next = next
        self.random = random

def serializeWithRandom(head):
    if not head:
        return ''
    node_to_idx = {}
    idx = 0
    cur = head
    while cur:
        node_to_idx[id(cur)] = idx
        idx += 1
        cur = cur.next
    parts = []
    cur = head
    while cur:
        r_idx = node_to_idx[id(cur.random)] if cur.random else -1
        parts.append(f'{cur.val},{r_idx}')
        cur = cur.next
    return '|'.join(parts)

def deserializeWithRandom(data):
    if not data:
        return None
    parts = data.split('|')
    nodes = [RandomNode(int(p.split(',')[0])) for p in parts]
    for i in range(len(nodes) - 1):
        nodes[i].next = nodes[i + 1]
    for i, p in enumerate(parts):
        r_idx = int(p.split(',')[1])
        nodes[i].random = nodes[r_idx] if r_idx != -1 else None
    return nodes[0]

n1, n2, n3 = RandomNode(1), RandomNode(2), RandomNode(3)
n1.next = n2
n2.next = n3
n1.random = n3
n2.random = n1
n3.random = None
serialized = serializeWithRandom(n1)
restored = deserializeWithRandom(serialized)
assert restored.val == 1
assert restored.random.val == 3
assert restored.next.random.val == 1
assert restored.next.next.random is None
print("All tests passed!")


## Hard 17 — Clone Graph Using Linked List Adjacency Representation


> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Clone an undirected graph where each node stores neighbors as a list (adjacency list style).


### Approach
BFS with a hash map from original node to its clone. For each neighbor, clone if not yet seen and add to the clone's neighbor list.

**Time:** O(V + E) | **Space:** O(V)

In [ ]:
class GraphNode:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []

def cloneGraph(node):
    if not node:
        return None
    from collections import deque
    clones = {node: GraphNode(node.val)}
    queue = deque([node])
    while queue:
        curr = queue.popleft()
        for neighbor in curr.neighbors:
            if neighbor not in clones:
                clones[neighbor] = GraphNode(neighbor.val)
                queue.append(neighbor)
            clones[curr].neighbors.append(clones[neighbor])
    return clones[node]

g1 = GraphNode(1)
g2 = GraphNode(2)
g3 = GraphNode(3)
g1.neighbors = [g2, g3]
g2.neighbors = [g1]
g3.neighbors = [g1]
clone = cloneGraph(g1)
assert clone is not g1
assert clone.val == 1
assert len(clone.neighbors) == 2
assert clone.neighbors[0].val == 2
assert clone.neighbors[1].val == 3
print("All tests passed!")


## Hard 18 — Implement Doubly Linked List with All Operations


> 🏢 **Asked by:** Amazon, Google
Implement a doubly linked list with `prepend`, `append`, `deleteHead`, `deleteTail`, and `toList`.


### Approach
Maintain `head`, `tail`, and `size`. Each node has `prev` and `next`. Use a dummy head and dummy tail to simplify edge cases.

**Time:** O(1) for head/tail ops | **Space:** O(n)

In [ ]:
class DLLNode:
    def __init__(self, val=0):
        self.val = val
        self.prev = None
        self.next = None

class DoublyLinkedList:
    def __init__(self):
        self.head = DLLNode()
        self.tail = DLLNode()
        self.head.next = self.tail
        self.tail.prev = self.head
        self.size = 0

    def _insert_between(self, val, before, after):
        node = DLLNode(val)
        node.prev = before
        node.next = after
        before.next = node
        after.prev = node
        self.size += 1

    def _delete_node(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev
        self.size -= 1

    def prepend(self, val):
        self._insert_between(val, self.head, self.head.next)

    def append(self, val):
        self._insert_between(val, self.tail.prev, self.tail)

    def deleteHead(self):
        if self.size:
            self._delete_node(self.head.next)

    def deleteTail(self):
        if self.size:
            self._delete_node(self.tail.prev)

    def toList(self):
        res = []
        cur = self.head.next
        while cur != self.tail:
            res.append(cur.val)
            cur = cur.next
        return res

dll = DoublyLinkedList()
dll.append(1)
dll.append(2)
dll.append(3)
assert dll.toList() == [1, 2, 3]
dll.prepend(0)
assert dll.toList() == [0, 1, 2, 3]
dll.deleteHead()
assert dll.toList() == [1, 2, 3]
dll.deleteTail()
assert dll.toList() == [1, 2]
print("All tests passed!")


## Hard 19 — Skiplist: Search, Add, Erase (LC 1206) — Alternative Implementation


> 🏢 **Asked by:** Amazon, Google
Design a skip list that supports `search(target)`, `add(num)`, and `erase(num)` in O(log n) expected time.


### Approach
Multi-level linked list. On `add`, randomly promote to higher levels with probability 0.5. Navigate from highest level down for `search`/`erase`.

**Time:** O(log n) expected per operation | **Space:** O(n log n)

In [ ]:
import random

class SkiplistNode:
    def __init__(self, val, level):
        self.val = val
        self.next = [None] * level

class Skiplist:
    MAX_LEVEL = 16

    def __init__(self):
        self.head = SkiplistNode(-float('inf'), self.MAX_LEVEL)
        self.level = 1

    def _random_level(self):
        lvl = 1
        while random.random() < 0.5 and lvl < self.MAX_LEVEL:
            lvl += 1
        return lvl

    def search(self, target):
        cur = self.head
        for i in range(self.level - 1, -1, -1):
            while cur.next[i] and cur.next[i].val < target:
                cur = cur.next[i]
        cur = cur.next[0]
        return cur is not None and cur.val == target

    def add(self, num):
        update = [self.head] * self.MAX_LEVEL
        cur = self.head
        for i in range(self.level - 1, -1, -1):
            while cur.next[i] and cur.next[i].val < num:
                cur = cur.next[i]
            update[i] = cur
        lvl = self._random_level()
        if lvl > self.level:
            for i in range(self.level, lvl):
                update[i] = self.head
            self.level = lvl
        node = SkiplistNode(num, lvl)
        for i in range(lvl):
            node.next[i] = update[i].next[i]
            update[i].next[i] = node

    def erase(self, num):
        update = [None] * self.MAX_LEVEL
        cur = self.head
        for i in range(self.level - 1, -1, -1):
            while cur.next[i] and cur.next[i].val < num:
                cur = cur.next[i]
            update[i] = cur
        target = cur.next[0]
        if not target or target.val != num:
            return False
        for i in range(self.level):
            if update[i].next[i] != target:
                break
            update[i].next[i] = target.next[i]
        return True

sl = Skiplist()
sl.add(1)
sl.add(2)
sl.add(3)
assert sl.search(0) == False
sl.add(4)
assert sl.search(1) == True
assert sl.erase(0) == False
assert sl.erase(1) == True
assert sl.search(1) == False
print("All tests passed!")


## Hard 20 — Implement Stack Using Linked List with getMin Operation


> 🏢 **Asked by:** Amazon, Google
Design a stack using a linked list that supports `push(x)`, `pop()`, `peek()`, `getMin()` all in O(1) time.


### Approach
Each linked list node stores both the value and the current minimum at time of push. `getMin()` reads the top node's stored minimum.

**Time:** O(1) per operation | **Space:** O(n)

In [ ]:
class MinStackNode:
    def __init__(self, val, min_val, next=None):
        self.val = val
        self.min_val = min_val
        self.next = next

class MinStack:
    def __init__(self):
        self.top_node = None

    def push(self, x):
        cur_min = min(x, self.top_node.min_val) if self.top_node else x
        self.top_node = MinStackNode(x, cur_min, self.top_node)

    def pop(self):
        if self.top_node:
            val = self.top_node.val
            self.top_node = self.top_node.next
            return val

    def peek(self):
        return self.top_node.val if self.top_node else None

    def getMin(self):
        return self.top_node.min_val if self.top_node else None

ms = MinStack()
ms.push(-2)
ms.push(0)
ms.push(-3)
assert ms.getMin() == -3
ms.pop()
assert ms.peek() == 0
assert ms.getMin() == -2
ms.push(1)
assert ms.getMin() == -2
print("All tests passed!")
